In [ ]:
import pandas as pd
import os
from highstreets import config
from highstreets.data_source_sink.dataloader import DataLoader
from highstreets.data_source_sink.datawriter import DataWriter
from highstreets.data_transformation.mcard_transform import McardTransform
from highstreets.core.sql_manager import SQLManager
from highstreets.data_transformation.mcard_weekly_processor import FileProcessor
from highstreets.api.clientbase import APIClient
from sqlalchemy import create_engine
import psycopg2
from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

base_dir = config.BASE_DIR
# initialize the database connection
database = os.getenv("PG_DATABASE")
username = os.getenv("PG_USER")
password = os.getenv("PG_PASSWORD")
host = os.getenv("PG_HOST")
port = os.getenv("PG_PORT")
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@" f"{host}:{port}/{database}"
)

# instantiate the classes
data_loader = DataLoader()
data_writer = DataWriter()
mcard_transform = McardTransform()
sql_manager = SQLManager()
api_client = APIClient()
dir_path = f"{base_dir}mastercard/sharefile_test"
mcard_weekly = FileProcessor(data_loader, data_writer, dir_path)

# Connect to PostgreSQL database
conn = psycopg2.connect(
    dbname=os.getenv("PG_DATABASE"),
    user=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
    host=os.getenv("PG_HOST"),
    port=os.getenv("PG_PORT"),
)

In [ ]:
print(base_dir)

In [ ]:
# Adjustment Factor generation using Spending Pulse
(adj_factor,
merged,
txn_inner_outer,
txn_io_monthly,
pulse_all,
pulse_pivot) = mcard_weekly.create_adjustment_factor(
    #table_name = "econ_busyness_mcard_raw_18_zoom",
    table_name="test_econ_busyness_mcard_inner_outer_txn",  # use new test table
    rolling_average_months=12,
    ffill_missing_dates=False,
    date_from=None,
    date_to=None,
    update_pg_table=True
)

In [ ]:
print(tuple(list(config.SECTORS_DF['geo_insights_raw'].unique())))

mcard_weekly_quad = data_loader.get_partial_data(
            table_name="econ_busyness_mcard_raw_18_zoom",
            columns=(
                "geo_name, segment, yr, wk, quad_id, weekday_weekend,"
                "industry, txn_amt"
            ),
            where_clause=f"yr = 2025"
            f" AND industry IN{tuple(list(config.SECTORS_DF['geo_insights_raw'].unique()))}"
            f" AND segment = 'Overall' AND geo_name='London'",
        )
mcard_weekly_quad

In [ ]:
mcard_weekly_quad[["yr", "wk"]] = mcard_weekly_quad[["yr", "wk"]].astype(
    int
)
# convert to a date column. The additional '1' sets the date as a Monday
Yw = (
    mcard_weekly_quad["yr"].astype(str)
    + mcard_weekly_quad["wk"].astype(str)
    + "1"
)
mcard_weekly_quad["week_start"] = pd.to_datetime(Yw, format="%G%V%w")
mcard_weekly_quad["quad_id"] = mcard_weekly_quad["quad_id"].astype("Int64")


In [ ]:
mcard_weekly_quad

## Plot adjustment factors

In [ ]:
# Load previous adjustments
adj_factor_pg = data_loader.get_full_data(table_name='econ_busyness_mcard_adjustment_factors')
adj_factor_full = pd.read_csv("Z:/hsds/data/mastercard/spendingpulse/mcard_adjustment_factor.csv")
adj_factor_full_test = pd.read_csv("Z:/hsds/data/mastercard/spendingpulse/test/mcard_adjustment_factor.csv")
adj_factor_s3 = pd.read_csv("s3://hsds-data/mastercard/spendingpulse/mcard_adjustment_factor.csv")
adj_factor_prev = pd.read_csv("Z:/HSDS/data/mastercard/SpendingPulse/test/mcard_adjustment_factor_20250331.csv")
adj_factor_from_raw = pd.read_csv("Z:/HSDS/data/mastercard/spendingpulse/test/mcard_adjustment_factor_20250501.csv")

adj_factor_full['date'] = pd.to_datetime(adj_factor_full['date'])
adj_factor_full_test['date'] = pd.to_datetime(adj_factor_full_test['date'])
adj_factor_s3['date'] = pd.to_datetime(adj_factor_s3['date'])
adj_factor_from_raw['date'] = pd.to_datetime(adj_factor_from_raw['date'])
adj_factor_prev['date'] = pd.to_datetime(adj_factor_prev['date'])

In [ ]:
adj_factor_pg

In [ ]:
adj_factor_s3

In [ ]:
adj_factor_full

In [ ]:
adj_factor_full_test

In [ ]:
#adj_factor_pg.to_csv("Z:/HSDS/data/mastercard/spendingpulse/test/mcard_adjustment_factor_20250331.csv",index=False)

In [ ]:
# Something seems to be changing in the last few motnths of the eating AF...?

In [ ]:
# Load towncentre table versions
tc_ds = pd.read_csv("Z:/HSDS/data/mastercard/mrli_3hourly/processed/towncentre/towncentre_3hourly_txn_2022-03-01_2025-03-31.csv")

tc_pg_aws = data_loader.get_full_data('aws_econ_busyness_mcard_towncentres_3hourly_txn')
tc_pg = data_loader.get_full_data('econ_busyness_mcard_towncentres_3hourly_txn')

In [ ]:
io_pg = data_loader.get_full_data('econ_busyness_mcard_inner_outer_txn')

In [ ]:
tc_pg['count_date'].max()

In [ ]:
tc_ds[(tc_ds['hours']=="'09-12")&(tc_ds['count_date']=="2025-02-28")&(tc_ds['tc_name']=="West End")]

In [ ]:
tc_pg[(tc_pg['hours']=="09-12")&(tc_pg['count_date']=="2025-02-28")&(tc_pg['tc_name']=="West End")]

In [ ]:
tc_pg_aws[(tc_pg_aws['hours']=="09-12")&(tc_pg_aws['count_date']=="2025-02-28")&(tc_pg_aws['tc_name']=="West End")]

In [ ]:
import matplotlib.pyplot as plt
tc_ds['count_date'] = pd.to_datetime(tc_ds['count_date'])
tc_pg_aws['count_date'] = pd.to_datetime(tc_pg_aws['count_date'])
tc_pg['count_date'] = pd.to_datetime(tc_pg['count_date'])


for area in tc_ds['tc_name'].unique()[:3]:
    ds_west_end = tc_ds[(tc_ds['hours']=="'09-12")&(tc_ds['tc_name']==area)]
    aws_west_end = tc_pg_aws[(tc_pg_aws['hours']=="09-12")&(tc_pg_aws['tc_name']==area)]
    pg_west_end = tc_pg[(tc_pg['hours']=="09-12")&(tc_pg['tc_name']==area)]


    plt.figure(figsize=(10,4))
    plt.plot(ds_west_end['count_date'],ds_west_end['txn_amt_adj'],label='DS')
    #
    plt.plot(aws_west_end['count_date'],aws_west_end['txn_amt_adj'],label='AWS')
    #plt.plot(pg_west_end['count_date'],pg_west_end['txn_amt_adj'],label='PG')
    plt.title(area)
    plt.legend()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt


for io in ['Inner','Outer']:
    # df = adj_factor[adj_factor['inner_outer']==io]
    df = adj_factor_pg[adj_factor_pg['inner_outer']==io]
    df1 = adj_factor_prev[adj_factor_prev['inner_outer']==io]    
    #df1 = adj_factor_full_test[adj_factor_full_test['inner_outer']==io]

    # df = adj_factor_from_raw[adj_factor_from_raw['inner_outer']==io]

    plt.figure(figsize=(10,4))
    plt.plot([df['date'].min(),df['date'].max()],[1,1],linestyle='--',color='lightgrey')

    plt.plot(df1['date'],df1['adjustment_factor_retail'],label='AF retail (previous)',color='cornflowerblue',zorder=100,linestyle='--')
    plt.plot(df1['date'],df1['adjustment_factor_eating'],label='eating (previous)',color='mediumpurple',linestyle='--')
    plt.plot(df1['date'],df1['adjustment_factor_apparel'],label='apparel (previous)',color='orangered',linestyle='--')


    plt.plot(df['date'],df['adjustment_factor_retail'],label='AF retail - PG',color='cornflowerblue')
    plt.plot(df['date'],df['adjustment_factor_eating'],label='eating - PG',color='mediumpurple')
    plt.plot(df['date'],df['adjustment_factor_apparel'],label='apparel - PG',color='orangered')

    plt.legend()   #plt.plot(df1['adjustment_factor_retail'])
    plt.ylim(ymin=0)
    plt.title(io)
    plt.show()

In [ ]:
# re-run from here
txn_inner_outer['geography'] = txn_inner_outer['geography'].replace({'Inner London':'Inner',
                                      'Outer London':'Outer'})
txn_inner_outer = pd.merge(txn_inner_outer,
                           adj_factor,
                           left_on=['geography','yr','month',],
                           right_on=['inner_outer','yr','month'],
                           how='left')

txn_inner_outer = txn_inner_outer.sort_values(by=['inner_outer','yr','month'])


pulse_all['geography'] = pulse_all['geography'].replace({'Inner London':'Inner',
                                      'Outer London':'Outer'})
pulse_pivot = pulse_all.pivot_table(index=['geography','yr','month','startDate'],
                                    columns='sector',
                                    values='Sales_inStore')
pulse_pivot = pd.merge(pulse_pivot,
                           adj_factor,
                           left_on=['geography','yr','month',],
                           right_on=['inner_outer','yr','month'],
                           how='left')

pulse_pivot = pulse_pivot.sort_values(by=['inner_outer','yr','month'])

In [ ]:

for ind in config.SECTORS_DF['geo_insights'].unique():
    txn_inner_outer[f'txn_amt_wd_{ind}_adj'] = txn_inner_outer[f'txn_amt_wd_{ind}']/txn_inner_outer[f'adjustment_factor_{ind}']
    txn_inner_outer[f'txn_amt_we_{ind}_adj'] = txn_inner_outer[f'txn_amt_we_{ind}']/txn_inner_outer[f'adjustment_factor_{ind}']

    txn_inner_outer = txn_inner_outer.groupby(
                            ["geography"], group_keys=False
                            ).apply(
                            lambda x: mcard_weekly.calculate_change_with_year_average(
                                x, yr_to_average=2018, col=f'txn_amt_wd_{ind}_adj'
                                        )
    )

    txn_inner_outer = txn_inner_outer.groupby(
                        ["geography"], group_keys=False
                        ).apply(
                        lambda x: mcard_weekly.calculate_change_with_year_average(
                            x, yr_to_average=2018, col=f'txn_amt_we_{ind}_adj'
                                    )
    )
    txn_inner_outer = txn_inner_outer.groupby(
                            ["geography"], group_keys=False
                            ).apply(
                            lambda x: mcard_weekly.calculate_change_with_year_average(
                                x, yr_to_average=2018, col=f'txn_amt_wd_{ind}'
                                        )
    )

    txn_inner_outer = txn_inner_outer.groupby(
                        ["geography"], group_keys=False
                        ).apply(
                        lambda x: mcard_weekly.calculate_change_with_year_average(
                            x, yr_to_average=2018, col=f'txn_amt_we_{ind}'
                                    )
    )
    pulse_pivot = pulse_pivot.groupby(
                        ["inner_outer"], group_keys=False
                        ).apply(
                        lambda x: mcard_weekly.calculate_change_with_year_average(
                            x, yr_to_average=2018, 
                            col=config.SECTORS_DF[
                                config.SECTORS_DF['geo_insights']==ind][
                                    'spending_pulse'].values[0]
                                    )
    )


### Comparing txn_innter_outer aggregated from RAW table vs new CLEAN test inner_outer table

In [ ]:
new_txn_inner_outer = data_loader.get_full_data("test_econ_busyness_mcard_inner_outer_txn")
# The week_start values are wrong??
new_txn_inner_outer = new_txn_inner_outer.sort_values(by=['inner_outer','week_start'])
new_txn_inner_outer = new_txn_inner_outer.groupby('inner_outer',group_keys=False).apply(lambda x: mcard_weekly.calculate_change_with_year_average(x,
                                                                      yr_to_average=2018,
                                                                      col='txn_amt_wd_retail'))
new_txn_inner_outer = new_txn_inner_outer.groupby('inner_outer',group_keys=False).apply(lambda x: mcard_weekly.calculate_change_with_year_average(x,
                                                                      yr_to_average=2018,
                                                                      col='txn_amt_we_retail'))
new_txn_inner_outer

In [ ]:
for sector in ['retail']:# config.SECTORS_DF['geo_insights'].unique():
    for io in ['Inner','Outer']:

        df_txn = txn_inner_outer[txn_inner_outer['geography']==io]
        new_txn_io = new_txn_inner_outer[new_txn_inner_outer['inner_outer']==io]

        plt.figure(figsize=(10,4))
        plt.plot([df_txn['startDate'].min(),df_txn['startDate'].max()],[1,1],linestyle='--',color='lightgrey')
        
        plt.plot(df_txn['startDate'],df_txn[f'txn_amt_wd_{sector}_change_from_2018'], label='Agg from raw')
        plt.plot(new_txn_io['week_start'],new_txn_io[f'txn_amt_wd_{sector}_change_from_2018'], label='New clean inner outer table')

        plt.legend()   #plt.plot(df1['adjustment_factor_retail'])
        plt.ylim(ymin=0)
        plt.title(io+f": {sector}")
        plt.show()

## Comparing Spending Pulse changes over time

In [ ]:


most_recent_file = mcard_weekly.get_most_recent_file(
            config.SP_DIR
        )



In [ ]:
sp_May_2025 = pd.read_csv(most_recent_file)
sp_May_2025['startDate'] = pd.to_datetime(sp_May_2025['startDate'],dayfirst=True)
sp_May_2025['endDate'] = pd.to_datetime(sp_May_2025['endDate'],dayfirst=True)
sp_May_2025 = sp_May_2025[sp_May_2025['endDate'].dt.is_month_end]
#sp_May_2025

In [ ]:
sp_s3 = pd.read_csv("s3://hsds-data/mastercard/spendingpulse/" f"SpendingPulse - London - 2018-2024.csv")
sp_z = pd.read_csv("z:/hsds/data/mastercard/spendingpulse/" f"SpendingPulse - London - 2018-2024.csv")

In [ ]:
sp_s3['startDate'] = pd.to_datetime(sp_s3['startDate'])
sp_s3['endDate'] = pd.to_datetime(sp_s3['endDate'])
sp_z['startDate'] = pd.to_datetime(sp_z['startDate'])
sp_z['endDate'] = pd.to_datetime(sp_z['endDate'])


In [ ]:
pre24_sp = pd.read_csv("Z:/HSDS/data/mastercard/SpendingPulse/Received/SpendingPulse - London - 2018-2024.csv")
pre24_sp['startDate'] = pd.to_datetime(pre24_sp['startDate'],dayfirst=True)
pre24_sp['endDate'] = pd.to_datetime(pre24_sp['endDate'],dayfirst=True)
pre24_sp = pre24_sp[(pre24_sp['endDate'].dt.is_month_end)&(pre24_sp['startDate']<'2024-01-01')]

adjusted_sp = pd.concat([pre24_sp,sp_May_2025])
#adjusted_sp


In [ ]:
sp_z.dtypes

In [ ]:
adjusted_sp.to_csv("z:/hsds/data/mastercard/spendingpulse/" f"SpendingPulse - London - 2018-2024.csv",index=False)
adjusted_sp.to_csv("s3://hsds-data/mastercard/spendingpulse/" f"SpendingPulse - London - 2018-2024.csv",index=False)

In [ ]:
import matplotlib.pyplot as plt

for sector in config.SECTORS_DF['geo_insights'].unique():
    for io in ['Inner','Outer']:

        sp = sp_s3[(sp_s3['geography']==io+' London')&(sp_s3['sector']==config.SECTORS_DF[
                                config.SECTORS_DF['geo_insights']==sector][
                                    'spending_pulse'].values[0])]
        spz = sp_z[(sp_z['geography']==io+' London')&(sp_z['sector']==config.SECTORS_DF[
                        config.SECTORS_DF['geo_insights']==sector][
                            'spending_pulse'].values[0])]

        # sp_adj = adjusted_sp[(adjusted_sp['geography']==io+' London')&(adjusted_sp['sector']==config.SECTORS_DF[
        #                         config.SECTORS_DF['geo_insights']==sector][
        #                             'spending_pulse'].values[0])]

        plt.figure(figsize=(10,4))
        
        # plt.plot(sp['date'],sp[f"""{config.SECTORS_DF[
        #                         config.SECTORS_DF['geo_insights']==sector][
        #                             'spending_pulse'].values[0]}"""],label='SP - currently used for AF')
        plt.plot(sp['startDate'],sp['Sales_inStore'],label='SP - S3')
        plt.plot(spz['startDate'],spz['Sales_inStore'],label='SP - Z drive')

        # plt.plot(sp_adj['startDate'],sp_adj['Sales_inStore'],label='SP - most recent version')

        plt.legend()   #plt.plot(df1['adjustment_factor_retail'])
        plt.ylim(ymin=0)
        plt.title(io+f": {sector}")
        plt.show()

## Plot Spending Pulse vs adjusted weekly spend

In [ ]:
for sector in config.SECTORS_DF['geo_insights'].unique():
    for io in ['Inner','Outer']:

        df_txn = txn_inner_outer[txn_inner_outer['geography']==io]
        sp = pulse_pivot[pulse_pivot['inner_outer']==io]

        plt.figure(figsize=(10,4))
        plt.plot([df_txn['startDate'].min(),df_txn['startDate'].max()],[1,1],linestyle='--',color='lightgrey')
        
        plt.plot(df_txn['startDate'],df_txn[f'txn_amt_wd_{sector}_change_from_2018'], label='MRLI')
        plt.plot(df_txn['startDate'],df_txn[f'txn_amt_wd_{sector}_adj_change_from_2018'],label='MRLI-adjusted')
        plt.plot(sp['date'],sp[f"""{config.SECTORS_DF[
                                config.SECTORS_DF['geo_insights']==sector][
                                    'spending_pulse'].values[0]}_change_from_2018"""],label='SP')

        plt.legend()   #plt.plot(df1['adjustment_factor_retail'])
        plt.ylim(ymin=0)
        plt.title(io+f": {sector}")
        plt.show()

In [ ]:
import numpy as np

for sector in config.SECTORS_DF['geo_insights'].unique():
    for io in ['Inner','Outer']:

        df_txn = txn_inner_outer[txn_inner_outer['geography']==io]
        sp = pulse_pivot[pulse_pivot['inner_outer']==io]

        plt.figure(figsize=(10,4))
        plt.plot([df_txn['startDate'].min(),df_txn['startDate'].max()],[1,1],linestyle='--',color='lightgrey')
        
        plt.plot(df_txn['startDate'],df_txn[f'txn_amt_wd_{sector}_change_from_2018'], label='MRLI')
        plt.plot(df_txn['startDate'],df_txn[f'txn_amt_wd_{sector}_adj_change_from_2018'],label='MRLI-adjusted')
        plt.plot(sp['date'],sp[f"""{config.SECTORS_DF[
                                config.SECTORS_DF['geo_insights']==sector][
                                    'spending_pulse'].values[0]}_change_from_2018"""],label='SP')

        plt.legend()   #plt.plot(df1['adjustment_factor_retail'])
        plt.ylim(ymin=0)
        plt.xlim(xmin=np.datetime64('2024-01-01'))

        plt.title(io+f": {sector}")
        plt.show()

## Plot Spending Pulse vs adjusted 3-hourly spend